This notebook explains and tests our post-hoc overtransfection estimation method.

How can we know to what extent an emperical dataset might be overtransfected? i.e. how many barcode collisions there are per cell?

Suppose we draw $m$ total MPRA barcodes per cell, uniformly from $M$ different MPRA barcodes. The expected number of unique is

$$E[Unique]=M\left[1-\left( \frac{M-1}{M} \right)^m \right]$$

[link](https://stats.stackexchange.com/questions/296005/the-expected-number-of-unique-elements-drawn-with-replacement)

We can measure the number of unique MPRA barcodes in each cell. We want to calculate $m$, actual number of transfected barcodes. Note that we are just assuming that observed unique is the expected value of unique, which is a big approximation.

$$\frac{E[Unique]}{M}=\left[1-\left( \frac{M-1}{M} \right)^m \right]$$

$$\left( \frac{M-1}{M} \right)^m=1-\frac{E[Unique]}{M}$$

$$  m=ln\left(1-\frac{E[Unique]}{M}\right)/ln\left(\frac{M-1}{M}\right)$$

That formula is used in the `Bounds` object, .

Let's test it with a bit of simulation. Let's begin by getting some ballpark figures :

In [55]:
import scMPRAforge as scm
import numpy as np
import pandas as pd
from itertools import product
from tqdm import tqdm
tqdm.pandas()

Ballpark total cells:

In [2]:
scm.SHENDURE_BOUNDS.cells_per_cell_type.sum()

43389

Ballpark total MPRA bc:

In [3]:
scm.SHENDURE_BOUNDS.total_uniq_mpra_bc

28300

Ballpark MOI:

In [4]:
scm.SHENDURE_BOUNDS.transfection_model.mu_nb

18.01055567079214

Define some functions:

In [33]:
def simple_transfection(bound,total_cells,library_size):
    #Similar logic to _simulate_single_replicate_transfection()

    #create barcodes
    cbc=scm.generate_barcodes(length=10, count=total_cells)
    mpra_bc=scm.generate_barcodes(length=10, count=library_size)
    #create ret df
    cells_df=pd.DataFrame({'cell_bc':cbc})
    #decide how many transfected into each cell
    cells_df["num_transfected"]=bound.transfection_model.draw_nb(len(cells_df))
    #repeat, so there is one row per tfection event
    cells_df=cells_df.loc[cells_df.index.repeat(cells_df["num_transfected"])].reset_index(drop=True)
    #drop num transfected, since it is no longer required
    cells_df=cells_df.drop(columns=["num_transfected"])
    #sample mprabc
    cells_df["mpra_bc"]=np.random.choice(mpra_bc, size=len(cells_df), replace=True)
    return cells_df

def emperical_collision_number(cell_df):
    cell_df=cell_df.groupby(["cell_bc"])["mpra_bc"].agg(total_plasmid="count",unique_plasmid="nunique")
    return cell_df["total_plasmid"].sum()-cell_df["unique_plasmid"].sum()

def estimated_collisions(cell_df):
    #drop duplicates, simulating collision process
    cell_df=cell_df.drop_duplicates()
    #estimate total mprabc
    estimated_total_uniq_mprabc=len(cell_df["mpra_bc"].unique())
    #get num unique mprabc transfected into each cell
    tfection=cell_df.groupby(["cell_bc"])["mpra_bc"].nunique().reset_index()
    tfection=tfection.rename({"mpra_bc":"unique_mpra_bc"},axis=1)
    #compute how many we think there actually were
    observed=tfection["unique_mpra_bc"]
    tfection["tot_plasmid"]=np.log(1-observed/estimated_total_uniq_mprabc)/np.log((estimated_total_uniq_mprabc-1)/estimated_total_uniq_mprabc)
    return tfection['tot_plasmid'].sum()-tfection["unique_mpra_bc"].sum()


Example of use...

In [34]:
toy=simple_transfection(bound=scm.SHENDURE_BOUNDS,
    total_cells=43389,
    library_size=28300)

print(f"Ground-truth collisions {emperical_collision_number(toy)}")
print(f"Estimated collisions {estimated_collisions(toy)}")


Ground-truth collisions 370
Estimated collisions 392.73136124468874


Now let's define a function which computes the correlation between estimation and actual for some regieme...

In [38]:
def estimation_goodness(total_cells,library_size,MOI,replicates,root_bounds=scm.SHENDURE_BOUNDS):
    synthetic_bounds=root_bounds.copy()
    synthetic_bounds.set_effective_moi(MOI)

    percent_errors=[]

    for _ in range(0,replicates):
        data=simple_transfection(bound=synthetic_bounds,
                            total_cells=total_cells,
                            library_size=library_size)
        
        gtruth=emperical_collision_number(data)
        estimate=estimated_collisions(data)
        percent_errors.append(np.abs((estimate-gtruth)/gtruth)*100)
    return np.mean(percent_errors)


Example of using the function:

In [39]:
estimation_goodness(total_cells=43389,library_size=28300,MOI=20,replicates=3)

6.8994745631221095

Now let's create a set of sets of reasonable experimental parameters

In [59]:
total_cells = [2000,5000,10000] #np.arange(1000, 10**6, 200000)
MOI = [5,10,20]#np.arange(5, 30, 5)
library_size = [20000,30000,40000]#np.arange(10000, 100000, 20000)

params = pd.DataFrame(
    product(total_cells, MOI, library_size),
    columns=["total_cells", "MOI", "library_size"]
)
params

,total_cells,MOI,library_size
0,2000,5,20000
1,2000,5,30000
2,2000,5,40000
3,2000,10,20000
4,2000,10,30000
5,2000,10,40000
6,2000,20,20000
7,2000,20,30000
8,2000,20,40000
9,5000,5,20000


Let's find out how well the estimator works for all of them

In [60]:
params["percent_error"] = df.progress_apply(
    lambda row: estimation_goodness(
        row["total_cells"],
        row["library_size"],
        row["MOI"],
        1,
        root_bounds=scm.SHENDURE_BOUNDS
    ),
    axis=1
)

100%|██████████| 810/810 [1:50:47<00:00,  8.21s/it]


In [61]:
params.to_csv("coupon_collector_cache.tsv",sep="\t")

In [62]:
params

,total_cells,MOI,library_size,percent_error
0,2000,5,20000,404.741650
1,2000,5,30000,124.248282
2,2000,5,40000,inf
3,2000,10,20000,inf
4,2000,10,30000,321.141247
5,2000,10,40000,308.922881
6,2000,20,20000,inf
7,2000,20,30000,297.070061
8,2000,20,40000,inf
9,5000,5,20000,123.308202


Looks like generally within a few fold of the real value. 
Ok actually I think I need to look at the specific figures, since a few fold could be a lot or a litle, depending... Also probably a few reps. But this is a tad slow. Maybe should just simulate all, then dump & summarize later.